In [ ]:
%md

# Healthcare Insurance Claim Approval Agent
%md
## 📄 Problem Statement

Modern healthcare insurance claim processing faces significant challenges in efficiency, accuracy, and scalability. The large volume and complexity of medical claims, each requiring nuanced interpretation of patient context and detailed policy rules, make manual review methods both time-consuming and inconsistent.

## 🎯 Objectives

The objective is to design an Advanced Agentic AI Claim Coverage Validation System using **LangGraph** and **LangChain** that should be able to:

- Parse and summarize semi-structured patient records and claim details using domain knowledge
- Retrieve and evaluate relevant insurance policy guidelines for specific claims
- Determine claim coverage by matching claim details to policy rules and requirements (diagnoses, procedures, age, gender and preauthorization)
- Generate clear, contextually accurate coverage decisions and provide reasoning for approvals or further routing for manual reviews
%md
## 🧠 Agent Architecture

The following diagrams illustrates the architecture of the single-agent Claim Coverage Agentic AI Validation System.
This is a **ReAct-style single agent created using LangGraph**. This ReAct Agent functions by leveraging a single agent, three tools, and a system instruction prompt.

The Agent works in below manner:

- The agent reads patient records and policy documents, then uses **summarize_patient_record** and **summarize_policy_guideline** tools to generate structured summaries.
- It then runs **check_claim_coverage** tool to evaluate whether the claimed procedure meets the coverage criteria.
- The agent produces a final decision - either **APPROVE** or **ROUTE FOR REVIEW**, along with a concise reason.

Finally, the Agentic System is being run on the records from test_records.json to generate the final output which is saved in a .csv file **submission.csv**

from IPython.display import Image
Image(filename='./Data/Agent_Architecture.png')
%md
## 🖥️ Initial Setup
%run ./.setup/learner_setup
%md
## 🗂️ Load Necessary Dependencies
import json
import csv
from langchain_openai import AzureChatOpenAI
from langchain_openai import AzureOpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.schema import Document
from langchain_core.tools import tool
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.prebuilt import create_react_agent
from IPython.display import display, Image, Markdown
from datetime import datetime
%md
## 🔐 Setup Authentication and LLM Client

The following code sets up the UAIS environment to authenticate the application and securely connect to the Azure OpenAI services. It retrieves the access token required for authorization and initializes both the LLM and embedding clients using Azure OpenAI authentication.
import os
from dotenv import load_dotenv
import httpx


# Function definition to generate access token
def get_access_token():
    auth = "https://api.uhg.com/oauth2/token"
    scope = "https://api.uhg.com/.default"
    grant_type = "client_credentials"

    with httpx.Client() as client:
        body = {
            "grant_type": grant_type,
            "scope": scope,
            "client_id": dbutils.secrets.get(scope="AIML_Training", key="client_id"),
            "client_secret": dbutils.secrets.get(
                scope="AIML_Training", key="client_secret"
            ),
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        resp = client.post(auth, headers=headers, data=body, timeout=60)
        access_token = resp.json()["access_token"]
        return access_token


# Fetch Environment Variables
load_dotenv("./Data/UAIS_vars.env")

AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
OPENAI_API_VERSION = os.environ["OPENAI_API_VERSION"]
EMBEDDINGS_DEPLOYMENT_NAME = os.environ["EMBEDDINGS_DEPLOYMENT_NAME"]
#MODEL_DEPLOYMENT_NAME = os.environ["MODEL_DEPLOYMENT_NAME"]
MODEL_DEPLOYMENT_NAME = "gpt-4.1-mini_2025-04-14"
PROJECT_ID = os.environ["PROJECT_ID"]

# Initialize Chat Client
chat_client = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=OPENAI_API_VERSION,
    azure_deployment=MODEL_DEPLOYMENT_NAME,
    temperature=0,
    azure_ad_token=get_access_token(),
    default_headers={"projectId": PROJECT_ID},
)

# Initialize Embedding Model
embeddings_client = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=OPENAI_API_VERSION,
    azure_deployment=EMBEDDINGS_DEPLOYMENT_NAME,
    azure_ad_token=get_access_token(),
    default_headers={"projectId": PROJECT_ID},
)

# Function to calculate age
# Input Parameters:
# 1. date_of_birth (e.g., "1982-03-15")
# 2. date_of_service (e.g., "2025-05-03")
# Returns: Age
# Logic: The age is computed by subtracting the birth year from the service year, and adjusting the result if the patient has not yet had their birthday in the service year. This calculation ensures the age reflects the last fully completed year, which is standard practice in healthcare.

# For example: If the date of birth is March 15, 1982 and the date of service is May 3, 2025, the computed age is 43.

def calculate_age(date_of_birth: str, date_of_service: str) -> int:
    """
    Calculate age based on date of birth and date of service.

    Parameters:
    - date_of_birth (str): Date of birth in 'YYYY-MM-DD' format.
    - date_of_service (str): Date of service in 'YYYY-MM-DD' format.

    Returns:
    - int: Age in completed years.
    """
    dob = datetime.strptime(date_of_birth, "%Y-%m-%d")
    dos = datetime.strptime(date_of_service, "%Y-%m-%d")

    age = dos.year - dob.year
    if (dos.month, dos.day) < (dob.month, dob.day):
        age -= 1
    return age
%md
## Load the necessary datasets needed to be used by the Agent in decision making

### 📝 Dataset Details

| Data File Name            | Description |
|----------------------|-------------|
| **`insurance_policies.json`** | This dataset contains the insurance policy details, including coverage rules, diagnosis-procedure mappings, age ranges, gender eligibility, preauthorization requirements, and additional notes as depicted in a sample policy record in the adjoining column. This data is used by the summarize_policy_guideline tool to retrieve the policy associated with a given ID and generates a structured summary of its contents. |
| **`reference_codes.json`**        | This dataset provides human-readable descriptions for CPT (procedure) codes and ICD-10 (diagnosis) codes. These mappings are used to enrich the output of your patient and policy summary tools. The mappings can be passed to the tools and referenced using LLM reasoning to improve clarity and explainability |

# Read insurance_policies.json dataset
with open('./Data/insurance_policies.json', 'r') as f:
    insurance_policies_data = json.load(f)

# Read reference_codes.json dataset
with open('./Data/reference_codes.json', 'r') as f:
    reference_codes_data = json.load(f)
# Apply TikToken for caching and set Telemetry to False to avoid logging
tiktoken_cache_dir = os.path.abspath("./.setup/tiktoken_cache/")
os.environ["TIKTOKEN_CACHE_DIR"] = tiktoken_cache_dir
os.environ["ANONYMIZED_TELEMETRY"] = "False"
%md
## Define LLM-Powered Tools

This agentic system uses three dedicated tools, each powered by an LLM, to support core agent tasks such as classification, retrieval, and response generation. These tools encapsulate distinct responsibilities and enable the agents to operate in a modular and interpretable manner.

### 📘 Tool Responsibilities

| Tool Name            | Description |
|----------------------|-------------|
| **`summarize_patient_record(record_str)`** | This tool is responsible for extracting a structured summary of a patient’s insurance claim record using LLM reasoning. It accepts a raw patient record (as a JSON string or plain string) and returns a well-structured summary report that is later be used for claim coverage evaluation. **Note : It uses ICD-10 and CPT code mappings (from reference_codes.json) to enrich the output with human-readable descriptions of medical codes.** |
| **`summarize_policy_guideline(policy_id)`**        | This tool is responsible for extracting a structured summary of a patient’s insurance claim record using LLM reasoning. It accepts a raw patient record (as a JSON string or plain string) and returns a well-structured summary report that will later be used for claim coverage evaluation. **Note : It uses ICD-10 and CPT code mappings (from reference_codes.json) to enrich the output with human-readable descriptions of medical codes.** |
| **`check_claim_coverage(record_summary, policy_summary)`**       | This tool determines whether the procedures claimed by a patient are covered under their insurance policy. It takes as input the structured summary of the patient record along with the corresponding policy summary generated by the earlier tools. The output of this tool is a claim approval decision i.e. either **"APPROVE"** or **"ROUTE FOR REVIEW"** with a brief explanation of the reasoning behind it. |

%md
## 🛠 summarize_patient_record tool

| Input            | Process         |  Output            |
|------------------|-----------------|--------------------|
| Raw JSON claim record | Looks into reference_codes_data dataset to retrieve the descriptions for Diagnosis Code and CPT Code. | Generates a summary that is clearly formatted and include the following seven labeled sections, in the specified order: Patient Demographics: It includes name, gender, and age (**Note: Age is precomputed using Python and included in the input record as "age"**), Insurance Policy ID, Diagnoses and Descriptions: Include ICD-10 codes and their mapped descriptions. Procedures and Descriptions: Include CPT codes and their mapped descriptions. Preauthorization Status: Clearly mention if preauthorization was required and whether it was obtained. Billed Amount (in USD)|


@tool
def summarize_patient_record(record_str: str) -> str:
    """
    Summarizes Patient Health Record for the give Patient Claim Record
    Searches the reference_codes_data dataset to find the description of diagnosis_codes and procedure_codes present in the given Patient Claim Record

    Parameters:
        record_str (str): JSON String that contains Patient's Claim Record

    Returns:
        str: Formatted String summarizing Patient Health Record
    """
    prompt = f"""    
    You are a helpful assistant that is expert in summarizing Patient Health Record.
    Your job is to create a summary of the patient's health record for given JSON String dataset {record_str}  as per the instructions given below:

    Given the following dataset:
    {record_str} and {reference_codes_data}
    
    - Extract and return description of diagnosis_codes and procedure_codes present in the {record_str}. 
    - Lookup to {reference_codes_data["ICD10"]} to fetch the description of diagnosis_codes.
    - Lookup to {reference_codes_data["CPT"]} to fetch the description of procedure_codes.

    The summary generated by this tool should be clearly formatted and include the following seven labeled sections, in the specified order:

        • Patient Demographics: Include name, gender, and age
        • Insurance Policy ID
        • Diagnoses and Descriptions: Include ICD-10 codes and their mapped descriptions.
        • Procedures and Descriptions: Include CPT codes and their mapped descriptions.
        • Preauthorization Status: Clearly mention if preauthorization was required and whether it was obtained.
        • Billed Amount (in USD)
        • Date of Service

    Instructions:
    - Carefully read the query and create the summary as mentioned.
    - Do not include any additional explanation or commentary.
    """

    # Run the prompt through the LLM client and return the result
    return chat_client.invoke(prompt).content.strip()
%md
## 🛠 summarize_policy_guideline tool

| Input            | Process         |  Output            |
|------------------|-----------------|--------------------|
| policy_id extracted from summarize_patient_record tool output  | Fetches Policy Details from insurance_policies_data dataset for input policy_id. Fetches Diagnosis Code and CPT Code Descriptions from reference_codes_data dataset. | Generates a well-formatted summary that outlines the specific claim coverage rules for each procedure under that policy. The summary generated by this tool clearly displays and include the following clearly labeled sections, in order: **Policy Details:** Include: policy ID and plan name, **Covered Procedures:** For each covered procedure listed in the policy, include the following sub-points: Procedure Code and Description (using CPT code mappings)Covered Diagnoses and Descriptions (using ICD-10 code mappings), Gender Restriction, Age Range, Preauthorization Requirement, Notes on Coverage (if any)|
@tool
def summarize_policy_guideline(policy_id: str) -> str:
    """
    Summarizes Insurance Policy Details for the given Insurance Policy ID
    Searches the insurance_policies_data dataset to find the policy details for the given Insurance Policy ID

    Parameters:
        policy_id (str): Insurance Policy ID

    Returns:
        str: Formatted String summarizing Insurance Policy Details
    """

    prompt = f"""
    You are a helpful assistant that retrieves insurance policy details from a JSON dataset and create a summary.
    
    Given the following dataset:
    {insurance_policies_data} and {reference_codes_data}

    - Search the dataset {insurance_policies_data} for a policy with policy_id {policy_id} and return all its attributes.
    - Lookup to {reference_codes_data["ICD10"]} to fetch the description of each covered_diagnoses.
    - Lookup to {reference_codes_data["CPT"]} to fetch the description of each procedure_code.

    The summary generated by this tool should be clearly formatted and include the following clearly labeled sections, in order:
            • Policy Details 
                - Policy ID
                - Plan name
            • Covered Procedures: For each procedure for the policy ID: {policy_id} in covered_procedures section, create a separate entry with required sub-points clearly listed below: 
                - Procedure Code and Descriptions
                - Covered Diagnoses and Descriptions
                - Gender Restriction
                - Age Range
                - Preauthorization Requirement
                - Notes on Coverage (if any)

    Instructions:
        - Create the summary as mentioned. Respond in a well structured format.
        - Do not include any additional explanation or commentary.

    """
    
    # Run the prompt through the LLM client and return the result
    return chat_client.invoke(prompt).content.strip()
%md
## 🛠 check_claim_coverage tool

| Input            | Process         |  Output            |
|------------------|-----------------|--------------------|
| record_summary from summarize_patient_record tool output & policy_summary from summarize_policy_guideline tool output  | It uses LLM-based reasoning with the help of prompts to evaluate each claimed procedure against the applicable policy conditions | Returns a coverage eligibility decision, either **APPROVE** or **ROUTING FOR REVIEW** for manual review by a human specialist. |

**Coverage Evaluation Criteria:** A procedure is approved only if all the following conditions are met:

- The patient's diagnosis code(s) match the policy-covered diagnoses for the claimed procedure.
- The procedure code is explicitly listed in the policy, and all associated conditions are satisfied.
- The patient's age falls within the policy's defined age range (inclusive of the lower bound, exclusive of the upper bound).
- The patient's gender matches the policy’s requirement for that procedure.
- If preauthorization is required by the policy, it must have been obtained.
- Only procedures and diagnoses explicitly listed in the patient record should be evaluated. For simplicity we have kept one procedure per patient.

If the above conditions are not satisfied, eligibility decision will be **ROUTING FOR REVIEW**.

For both decisions, tool will provide a concise response to support the decision taken
@tool
def check_claim_coverage(record_summary: str, policy_summary: str) -> str:
    """
    Determines whether the procedures claimed by a patient are covered under their insurance policy.
    Evaluates each claimed procedure against the applicable policy conditions and returns a coverage eligibility decision

    Parameters:
        record_summary (str): Patient Record Summary
        policy_summary (str): Insurance Policy Summary

    Returns:
        str: Formatted String that contains:
        Decision: Either APPROVE or ROUTE FOR REVIEW
        Reason: Concise reason to support the decision
    """

    prompt = f"""
    You are an expert Healthcare Insurance Claim Approval Agent that helps in determining whether the procedures claimed by a patient are covered under their insurance policy.
    
    Only procedures and diagnoses explicitly listed in the {record_summary} should be evaluated.
    Return a Decision either APPROVE or ROUTE FOR REVIEW along with a concise reason to support the decision.
    Display complete description of CPT Codes and Diagnosis Codes in the Reason.

    The decision should be APPROVE only if:
    - CPT Code in {record_summary} matches with the Procedure Code and Descriptions in the {policy_summary}
    - ICD-10 Code in {record_summary} is available in the Covered Diagnoses and Descriptions for the Procedure Code in the {policy_summary}    
    - Age in {record_summary} falls within the Age Range defined for the Procedure Code in {policy_summary}
    - The patient's gender in {record_summary} matches the policy’s requirement for the procedure code in {policy_summary}.
    - If Preauthorization Required is Yes for the CPT Code in {policy_summary}, Preauthorization Obtained is Yes in the {record_summary}

    If the above conditions are not met, the decision should be ROUTE FOR REVIEW.

    If decision is APPROVE, refer to the below samples to generate the appropriate reason.
    
    Sample 1: The claim for the office visit (CPT code 99213) has been approved. This procedure is covered under the policy for the patient’s diagnosis of essential hypertension (I10). The patient, a 17-year-old male, meets all eligibility criteria, and no preauthorization is required for this service.

    Sample 2: The claim for collecting venous blood by venipuncture (CPT code 36415) has been approved. This procedure is covered under the policy for the patient’s diagnosis of low back pain (M54.5). The patient is 14 years old, which falls within the covered age range of 11 to 27, and there are no gender-based restrictions. Preauthorization isn’t required, but it was obtained, and all policy requirements have been satisfied.

    If the decision is ROUTE FOR REVIEW, refer to the below samples to generate the appropriate reason:

    Sample 1: The claim for the chest X-ray (CPT code 71020) cannot be automatically approved, as the policy only covers this procedure for the diagnosis of low back pain (M54.5). The patient’s diagnosis of gastro-esophageal reflux disease (K21.9) does not align with the policy’s covered conditions. Although the patient's age and gender meet the policy requirements, the diagnosis does not match the criteria for coverage. Therefore, the claim needs to be routed for further manual review.

    Sample 2: The claim for the Hemoglobin glycosylated A1c test (CPT code 83036) cannot be automatically approved, as the patient’s age of 55 falls outside the allowed age range of 34 to 44 specified in the policy. Although the diagnosis (K21.9 – Gastro-esophageal reflux disease without esophagitis) is covered, the age requirement is not met. Therefore, the claim needs to be routed for further manual review.
    
    Instructions:        
        - Do not include any additional explanation or commentary.
        - Make sure to include the Procedure Code and  Diagnosis Code in the Reason.
    """

    # Run the prompt through the LLM client and return the result
    return chat_client.invoke(prompt).content.strip()
%md
## 🔗 Bind Tools for AI Agent to LLM
# Create the list of tools in the workflow
tools = [summarize_patient_record,summarize_policy_guideline,check_claim_coverage]

# Bind the tools to the LLM so it can invoke them when necessary
# Enables tool-calling functionality in LangChain
llm_with_tools = chat_client.bind_tools(tools=tools)
%md
## 📝 Define Agent Instructions Prompt

These are the prompts that will help Agent in determining: 
  - What steps to execute
  - Which tools to be called in which sequence
  - Finally what result needs to be produced as the output and in which format.
# Instruction prompt for the overall Agent
AGENT_PROMPT_TXT = """You are an agent designed to act as an expert in Healthcare Insurance Claim Approval process.

Given a user query, execute below steps in sequence:

1. Your job is to create a structured summary of the patient’s insurance claim record using below instructions:

    - Call summarize_patient_record tool to generate a structured summary of the patient’s insurance claim record.
    - Gracefully exit from the workflow if the Policy ID is not found.

2. Your job is to create a structured summary of the insurance policy using the below instructions:
    
    - Call summarize_policy_guideline tool to genarate a structured summary of the insurance policy.
    - Generate the structured summary of the insurance policy only if the Policy ID is found.
    - Gracefully exit from the workflow if the Policy ID is not found.

3. Your job is to provide the decision either APPROVE or ROUTE FOR REVIEW, along with a concise reason using below instructions:
    
    - Call check_claim_coverage tool    
    - Generate the final response exactly in the below format:
        Decision: 
        Reason:

Instructions:
    - Politely decline to answer any queries not related to medical or healthcare information.

"""
AGENT_SYS_PROMPT = SystemMessage(content=AGENT_PROMPT_TXT)
%md
## 📜 Define Agent State Schema

We maintain Agent's state using a structured list of messages.

This state defines what data is carried forward between each step in the agent's reasoning and decision-making process.

Here, we define a simple state using Python’s TypedDict with one key field:

messages: A list of all messages in the Agent so far. This includes the user query, LLM responses, any tool call requests, and the tool responses
The messages list is annotated using add_messages, which ensures that every new message is appended properly as the agent moves through each step in the graph.

This schema is later passed to the LangGraph StateGraph, enabling the agent to track all reasoning and tool usage as a growing message history.

It forms the foundation for the **ReAct loop — Reason, Act (tool call), Observe, Repeat** — that we build using LangGraph.
# Define the agent's state schema for storing the message history
class State(TypedDict):
    messages: Annotated[list, add_messages]
%md
## ⚙️ Create Tool-Use AI Agent

The Agent is built manually using StateGraph from LangGraph as the full ReAct-style agent 

**Define the LLM reasoning step:**
tool_calling_llm() is the function used to invoke the LLM at each step
It adds the system prompt (AGENT_SYS_PROMPT) to the current message history and queries the LLM using llm_with_tools.invoke(...)
The response (which may be a tool call request or final answer) is returned and stored as the updated messages

**Initialize the graph:**
A StateGraph is built using our State schema

**Add nodes:**
"tool_calling_llm" is the planning and reasoning node where the agent decides what to do next "tools" is a ToolNode, which handles execution of whichever tool the LLM requested

**Add routing logic:**
We start the graph from "tool_calling_llm"
We use add_conditional_edges(...) to route based on whether the LLM output is a tool call request using the tools_condition utility function:
  - If it is a tool call request, we go to the tools node to execute the requested function
 - If not, we route to END, completing the agent's task
# Create the node function that handles reasoning and planning using the LLM
def tool_calling_llm(state: State) -> State:
    # Extract the current conversation history from the state
    current_state = state["messages"]

    # Prepend the system instructions to the current message history
    state_with_instructions = [AGENT_SYS_PROMPT] + current_state

    # Call the LLM to generate a new message (either a response or a tool call request)
    response = [llm_with_tools.invoke(state_with_instructions)]

    # Return the updated state containing the new message
    return {"messages": response}

# Build the graph
builder = StateGraph(State)

# Add nodes
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools=tools))

# Add edges
builder.add_edge(START, "tool_calling_llm")
# Conditional edge
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition,  # conditional routing function
    # If the latest message (result) from LLM is a tool call request -> tools_condition routes to tools
    # If the latest message (result) from LLM is a not a tool call -> tools_condition routes to END
    ["tools", END],
)
builder.add_edge(
    "tools", "tool_calling_llm"
)  # this is the key feedback loop in the agentic system

# Compile Agent Graph
claimIQ_agent = builder.compile()
%md
## 🧠 Agent Name: claimIQ
### Agent workflow:

Below the claimIQ Agent workflow:

**Start → Tool Call (Summarize Record) → Tool Call (Summarize Policy) → Tool Call (Validate Eligibility) → Agent's Final Decision with concise Reasoning → End**
from IPython.display import Image
Image(filename='./Data/tool_use_agent_arch.png', height=200, width=300)
%md
## Build Utility Function to Stream Agent Results

Helper function to stream the step-by-step output of the agent. This is helpful  for tracing the Agent flow:
- What the agent doing in each step
- Which tool it decides to call
- What response it gets from that tool
- How it forms the final reply
# Utility function to call the agent and stream its step-by-step reasoning
def call_agent(agent, query, verbose=False):

    # Stream the agent's execution for the given query
    for event in agent.stream(
        {"messages": [HumanMessage(content=query)]},  # Input prompt
        stream_mode='values'  # Stream output as intermediate values
    ):
        # If verbose is enabled, print each intermediate message
        if verbose:
            event["messages"][-1].pretty_print()

    # Return the final message content for optional downstream use
    return event["messages"][-1].content
%md
## 🧪 Test out Agent!

Run a complete test of Tool-Use AI Agent by passing the records from **test_records.json** file

  - The test_records.json file contains 10 patient claim records
  - These records are run through the agent and final response from Agent is recorded for each claim.
  - Results are stored in a file named **submission.csv** for each claim record
  - The result file contains two columns **patient_id** and **generated_response**, corresponding to each patient in the test dataset
  - The generated_response column is populated with the agent’s output (**Decision and Reason**) for that specific patient ID.
# Filename for the CSV file
csv_filename = "./submission.csv"

# Define header row
header_row = ["patient_id", "generated_response"]

# Read test_records.json dataset
with open("./Data/test_records.json", "r") as f:
    test_claims_dataset = json.load(f)

# Open the file in write mode ('w') with newline='' to prevent blank rows
with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
    # Create a csv.writer object
    writer = csv.writer(file)

    # Write the header row
    writer.writerow(header_row)

    # Write the data rows
    # Loop through each record in order to process that through the AI Agent
    for claim_record in test_claims_dataset:
        patient_id = claim_record["patient_id"]
        print("Processing patient claim record for patient_id: ", {patient_id})
        query = json.dumps(claim_record, indent=2)

        # Call the agent to process the claim record
        generated_response = call_agent(claimIQ_agent, query, verbose=True)

        # Add "- " before 'Decision' and 'Reason' as mentioned in the guidelines to match csv file output format
        generated_response = generated_response.replace("Decision:", "- Decision:").replace("Reason:", "- Reason:")

        # Write the results to the file
        data_row = [patient_id, generated_response]
        writer.writerow(data_row)

print(f"\n\n CSV file '{csv_filename}' created and data written successfully.")